# ANLP — Session 1b: BART merger retraining

**What this run does** (vs Session 1):
- BART is now trained as the *merge step* of the chunk+aggregate pipeline.
  Pre-generates per-chunk summaries with pretrained BART, concatenates them
  (~960 tokens for ~30 chunks), and trains BART to map that intermediate
  summary to the gold reference. Matches what BART actually does at inference.

**Estimated time:** ~1 hour on T4 (10 min generate chunk-summary cache + ~17 min train + ~10 min BART inference).

Skips the prompting conditions — those are already done from Session 1.

In [ ]:
import os
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
WORK = '/kaggle/working'
REPO = f'{WORK}/ANLP_Project'
if not os.path.isdir(REPO):
    !git clone --quiet --branch setup/local-run https://github.com/christiandalfarra/ANLP_Project.git $REPO
%cd $REPO
!git pull --quiet

In [ ]:
!pip install -q sentencepiece bert_score rouge_score 'transformers>=4.40' 'accelerate>=0.30'

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## Wipe any old BART checkpoint & prediction file before re-training

In [ ]:
!rm -rf checkpoints/bart
!rm -f outputs/predictions/finetuned_bart.json

## Fine-tune BART (merger style) — generates chunk-summary cache then trains

In [ ]:
!python scripts/run_finetuning.py --model bart --output_dir checkpoints/bart

## BART inference on 9 test matches

In [ ]:
!python scripts/run_inference_finetuned.py --model bart --checkpoint checkpoints/bart

## Evaluate (ROUGE-only)

In [ ]:
!python scripts/run_evaluation.py --no-bertscore

In [ ]:
import os, json, glob
for p in sorted(glob.glob('outputs/predictions/*.json')):
    print('===', os.path.basename(p))
    d = json.load(open(p))
    mid, summary = next(iter(d.items()))
    print(f'[{mid}]\n{summary[:600]}\n')
for p in sorted(glob.glob('outputs/results/*')):
    print(p)
    if p.endswith('.csv'):
        print(open(p).read())